# 3. Inference and submission
Fit the pair model on the labelled training candidates, apply the validation-selected threshold to test candidates, and write both required TSV files. No external data or identity lookup is used.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dataset').exists(): REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'code/business_entity_resolution'))
from src.entity_resolution import prepare_frame, make_pairs, labelled_pairs, train_pair_model, write_submission

MAX_ROWS = None  # set to a small integer for a smoke test
train_dir = REPO_ROOT / 'dataset/train'
test_dir = REPO_ROOT / 'dataset/test'
read = lambda folder, name: pd.read_csv(folder / name, sep='\t', nrows=MAX_ROWS)
s1_train = prepare_frame(read(train_dir, 'train_source1.tsv'))
targets_train = prepare_frame(pd.concat([read(train_dir, 'train_source2.tsv'), read(train_dir, 'train_source3.tsv')], ignore_index=True))
truth = read(train_dir, 'train_ground_truth.tsv')

In [ ]:
train_pairs, _ = make_pairs(s1_train, targets_train, max_candidates=250)
labelled, labels = labelled_pairs(train_pairs, truth, negative_ratio=5)
model = train_pair_model(labelled, labels)

# Keep this value from 02_pair_features_and_validation.ipynb after validation.
THRESHOLD = 0.72

s1_test = prepare_frame(read(test_dir, 'test_source1.tsv'))
targets_test = prepare_frame(pd.concat([read(test_dir, 'test_source2.tsv'), read(test_dir, 'test_source3.tsv')], ignore_index=True))
test_pairs, candidate_map = make_pairs(s1_test, targets_test, max_candidates=250)
test_probabilities = model.predict_proba(test_pairs.iloc[:, 2:])[:, 1] if len(test_pairs) else []
output_dir = REPO_ROOT / 'output'
output_dir.mkdir(exist_ok=True)
write_submission(s1_test, test_pairs, test_probabilities, THRESHOLD, candidate_map, str(output_dir / 'matching_results.tsv'), str(output_dir / 'candidate_pairs.tsv'))
print('wrote', output_dir / 'matching_results.tsv')
print('wrote', output_dir / 'candidate_pairs.tsv')

In [ ]:
# Run from the repository root after the cells above.
import subprocess
subprocess.run([sys.executable, str(REPO_ROOT / 'utils/validate_submission.py'), '--matching', str(output_dir / 'matching_results.tsv'), '--candidate', str(output_dir / 'candidate_pairs.tsv'), '--test-dir', str(test_dir)], check=False)